<a href="https://colab.research.google.com/github/Ziqi-Li/GIS5106/blob/main/notebooks/W12_sam3_with_LLM_prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W12 Segmenting remote sensing imagery with text prompts and the Segment Anything Model 3 (SAM 3)

This notebook shows how to generate object masks from text prompts with the Segment Anything Model (SAM) 3 model.

More details of SAM 3 can be found at here: https://docs.ultralytics.com/models/sam-3/#segment-with-text-prompts


SAM 3 is a gated free access model, its access needs to be applied from hugging face: https://huggingface.co/facebook/sam3

They usually respond to your request within 1 day.

Make sure you use GPU runtime for this notebook. For Google Colab, go to `Runtime` -> `Change runtime type` and select `GPU` as the hardware accelerator. `CPU` is also fine but will be slow.

In [ ]:
%pip install leafmap localtileserver

In [ ]:
import leafmap


## Create an interactive map

centered at FSU.

In [ ]:
m = leafmap.Map(center=[30.44179010929151, -84.2976182150657], zoom=19, height="800px")

m.add_basemap("SATELLITE")

m

## Download a sample image

Pan and zoom the map to select the area of interest. Use the draw tools to draw a polygon or rectangle on the map.

Alternatively, you can also assign a bounding box.

In [ ]:
bbox = m.user_roi_bounds()

#alternatively you can manually specify one:
if bbox is None:
    bbox = [-84.29809675855033, 30.443524831567117, -84.29320586964847, 30.439618789531707]

In [ ]:
image = "Image.tif"# output file for the specified image.

leafmap.map_tiles_to_geotiff(
    output=image, bbox=bbox, zoom=19, source="Satellite", overwrite=True
)

Display the downloaded image on the map.

In [ ]:
m.layers[-1].visible = False

m.add_raster(image, layer_name="Image")
m

## Initialize LangSAM class

The initialization of the LangSAM class might take a minute. The initialization downloads the model weights and sets up the model for inference.

In [ ]:
pip install -U ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Upload the `sam3.pt` downloaded from huggingface to Google drive. Then connect with your drive and put the path to the model here.

In [ ]:
from ultralytics.models.sam import SAM3SemanticPredictor

# Initialize predictor with configuration
overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model="/content/drive/MyDrive/sam3.pt",
    half=True,  # Use FP16 for faster inference
    save=True,
)
predictor = SAM3SemanticPredictor(overrides=overrides)


### Normal Image

In [ ]:
# Set image once for multiple queries
path = "https://media.istockphoto.com/id/158353604/photo/birthday-party.jpg?s=612x612&w=0&k=20&c=c-bEbjCNZonZd30xBIPtqsMFpbxUb9kdSUlZb2OW7yA="

predictor.set_image(path)

results_normal = predictor(text=["the child with party hat"])

results_normal[0].save(filename="party hat.jpg")


### Remote Sensing Image

In [ ]:

# Set image once for multiple queries
predictor.set_image("Image.tif")

# Query with multiple text prompts
results = predictor(text=["buildings"])
results[0].save(filename="buildings.jpg")

#Works with descriptive phrases
results2 = predictor(text=["buildings with red roof"])
results2[0].save(filename="buildings_red_roof.jpg")


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# 1. Run the prediction
results = predictor(text=["buildings"])

# 2. Define your threshold
threshold = 0.5

for result in results:
    # Original image for plotting
    img_bgr = result.orig_img
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Initialize a combined mask for display
    combined_mask = np.zeros(img_rgb.shape[:2], dtype=np.uint8)

    # Check if masks exist
    if result.masks is not None:
        # result.boxes.conf contains the confidence scores
        confs = result.boxes.conf.cpu().numpy()
        masks = result.masks.data.cpu().numpy()

        for i, conf in enumerate(confs):
            if conf >= threshold:
                # Merge masks that pass the threshold
                combined_mask = np.maximum(combined_mask, masks[i])

    # 3. Display using Matplotlib
    plt.figure(figsize=(12, 8))

    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(img_rgb)
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title(f"Filtered Masks (Conf > {threshold})")
    plt.imshow(combined_mask, cmap='gray')
    plt.axis('off')

    plt.show()

### Red Roof

In [ ]:

# 1. Run the prediction
results = predictor(text=["buildings with red roof"])

# 2. Define your threshold
threshold = 0.8

for result in results:
    # Original image for plotting
    img_bgr = result.orig_img
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Initialize a combined mask for display
    combined_mask = np.zeros(img_rgb.shape[:2], dtype=np.uint8)

    # Check if masks exist
    if result.masks is not None:
        # result.boxes.conf contains the confidence scores
        confs = result.boxes.conf.cpu().numpy()
        masks = result.masks.data.cpu().numpy()

        for i, conf in enumerate(confs):
            if conf >= threshold:
                # Merge masks that pass the threshold
                combined_mask = np.maximum(combined_mask, masks[i])

    # 3. Display using Matplotlib
    plt.figure(figsize=(12, 8))

    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(img_rgb)
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title(f"Filtered Masks (Conf > {threshold})")
    plt.imshow(combined_mask, cmap='gray')
    plt.axis('off')

    plt.show()